In [ ]:
# Step 1: Install required dependencies
%pip install -q kagglehub pandas

In [17]:
import os
import shutil
import kagglehub
import pandas as pd

# Step 2: Configure Destination Directories on Google Drive
try:
    from google.colab import drive
    print("Detected Google Colab environment. Mounting Google Drive...")
    drive.mount("/content/drive")
    DEST_DIR = "/content/drive/MyDrive/capstone/foodRecSys-v1"
except ImportError:
    print("Running in local environment.")
    DEST_DIR = os.path.abspath("mydrive/capstone/foodRecSys-v1")

# Dedicated subfolder specifically for all CSV files
CSV_DIR = os.path.join(DEST_DIR, "csv")

os.makedirs(DEST_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

print(f"✓ Root Dataset Directory: {DEST_DIR}")
print(f"✓ Dedicated CSV Directory: {CSV_DIR}")

Detected Google Colab environment. Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Root Dataset Directory: /content/drive/MyDrive/capstone/foodRecSys-v1
✓ Dedicated CSV Directory: /content/drive/MyDrive/capstone/foodRecSys-v1/csv


In [ ]:
# Step 3: Download dataset from Kaggle using kagglehub
print("Downloading FoodRecSys-V1 dataset from Kaggle (elisaxxygao/foodrecsysv1)...")
downloaded_path = kagglehub.dataset_download("elisaxxygao/foodrecsysv1")
print(f"Dataset downloaded to cache at: {downloaded_path}\n")

# Step 4: Prioritize saving all CSV files directly into Google Drive
print(f"=== Saving all CSV files directly to Google Drive ===")
csv_files_copied = []

for root, dirs, files in os.walk(downloaded_path):
    for f in files:
        if f.lower().endswith(".csv"):
            src_file = os.path.join(root, f)
            size_mb = os.path.getsize(src_file) / (1024 * 1024)
            
            # 1. Save directly into root DEST_DIR
            dst_root = os.path.join(DEST_DIR, f)
            shutil.copy2(src_file, dst_root)
            
            # 2. Save also into dedicated CSV_DIR
            dst_csv = os.path.join(CSV_DIR, f)
            shutil.copy2(src_file, dst_csv)
            
            csv_files_copied.append((f, size_mb, dst_csv))
            print(f"  ✓ Saved CSV: {f:<30} ({size_mb:.2f} MB) -> {dst_csv}")

# Ensure all file writes are flushed to disk/Drive
try:
    os.sync()
except AttributeError:
    pass

print(f"\nSuccessfully transferred {len(csv_files_copied)} CSV file(s) to Google Drive!")

In [ ]:
# Step 4b (Optional): Transfer Image Folders to Google Drive
# Image folders contain tens of thousands of image files (>5GB).
# Set COPY_IMAGES = True to copy images as well, or False to keep only CSVs.
COPY_IMAGES = True

if COPY_IMAGES:
    print(f"\nTransferring image folders to {DEST_DIR}...")
    for item in os.listdir(downloaded_path):
        src_item = os.path.join(downloaded_path, item)
        dst_item = os.path.join(DEST_DIR, item)
        
        if os.path.isdir(src_item):
            print(f"  Copying folder '{item}' to Drive (this may take a few minutes)...\n  {src_item} -> {dst_item}")
            shutil.copytree(src_item, dst_item, dirs_exist_ok=True)
            print(f"  ✓ [Folder copied] {item}")
else:
    print("Skipping image folders copy. Set COPY_IMAGES = True if images are needed in Drive.")

In [18]:
# Step 5: Load, inspect & confirm Recipe Data is saved in Drive
recipe_file = os.path.join(CSV_DIR, "core-data_recipe.csv")
if not os.path.exists(recipe_file):
    recipe_file = os.path.join(DEST_DIR, "core-data_recipe.csv")

if os.path.exists(recipe_file):
    df_recipe = pd.read_csv(recipe_file)
    print(f"=== Recipes Dataset Loaded Successfully ===")
    print(f"File in Drive: {recipe_file}")
    print(f"Shape: {df_recipe.shape[0]:,} rows x {df_recipe.shape[1]} columns")
    print(f"Columns: {list(df_recipe.columns)}\n")
    
    # Explicitly guarantee saved as CSV in Drive
    save_path = os.path.join(CSV_DIR, "core-data_recipe.csv")
    df_recipe.to_csv(save_path, index=False)
    print(f"✓ Confirmed saved to Drive: {save_path}")
    
    try:
        from IPython.display import display
        display(df_recipe.head())
    except ImportError:
        print(df_recipe.head())
else:
    print(f"Notice: '{recipe_file}' was not found.")

=== Recipes Dataset Loaded Successfully ===
File in Drive: /content/drive/MyDrive/capstone/foodRecSys-v1/core-data_recipe.csv
Shape: 45,630 rows x 6 columns
Columns: ['recipe_id', 'recipe_name', 'image_url', 'ingredients', 'cooking_directions', 'nutritions']

✓ Confirmed saved to Drive: /content/drive/MyDrive/capstone/foodRecSys-v1/csv/core-data_recipe.csv


,recipe_id,recipe_name,image_url,ingredients,cooking_directions,nutritions
0,240488,"Pork Loin, Apples, and Sauerkraut",https://images.media-allrecipes.com/userphotos...,sauerkraut drained^Granny Smith apples sliced^...,{'directions': u'Prep\n15 m\nCook\n2 h 30 m\nR...,"{u'niacin': {u'hasCompleteData': False, u'name..."
1,218939,Foolproof Rosemary Chicken Wings,https://images.media-allrecipes.com/userphotos...,chicken wings^sprigs rosemary^head garlic^oliv...,"{'directions': u""Prep\n20 m\nCook\n40 m\nReady...","{u'niacin': {u'hasCompleteData': True, u'name'..."
2,87211,Chicken Pesto Paninis,https://images.media-allrecipes.com/userphotos...,focaccia bread quartered^prepared basil pesto^...,{'directions': u'Prep\n15 m\nCook\n5 m\nReady ...,"{u'niacin': {u'hasCompleteData': True, u'name'..."
3,245714,Potato Bacon Pizza,https://images.media-allrecipes.com/userphotos...,red potatoes^strips bacon^Sauce:^heavy whippin...,{'directions': u'Prep\n20 m\nCook\n45 m\nReady...,"{u'niacin': {u'hasCompleteData': True, u'name'..."
4,218545,Latin-Inspired Spicy Cream Chicken Stew,https://images.media-allrecipes.com/userphotos...,skinless boneless chicken breast halves^diced ...,{'directions': u'Prep\n10 m\nCook\n8 h 15 m\nR...,"{u'niacin': {u'hasCompleteData': False, u'name..."


In [ ]:
# Step 6: Load, inspect & confirm Interaction/Rating Data is saved in Drive
rating_file = os.path.join(CSV_DIR, "core-data-train_rating.csv")
if not os.path.exists(rating_file):
    rating_file = os.path.join(DEST_DIR, "core-data-train_rating.csv")

if os.path.exists(rating_file):
    df_ratings = pd.read_csv(rating_file)
    print(f"=== Ratings Dataset Loaded Successfully ===")
    print(f"File in Drive: {rating_file}")
    print(f"Shape: {df_ratings.shape[0]:,} rows x {df_ratings.shape[1]} columns")
    print(f"Columns: {list(df_ratings.columns)}\n")
    
    # Explicitly guarantee saved as CSV in Drive
    save_path = os.path.join(CSV_DIR, "core-data-train_rating.csv")
    df_ratings.to_csv(save_path, index=False)
    print(f"✓ Confirmed saved to Drive: {save_path}")
    
    try:
        from IPython.display import display
        display(df_ratings.head())
    except ImportError:
        print(df_ratings.head())
else:
    print(f"Notice: '{rating_file}' was not found.")

In [ ]:
# Step 7: Final Verification of all CSV Files Stored in Google Drive
print("=== Final Verification of CSV Files in Google Drive ===\n")

csv_folder = CSV_DIR if os.path.exists(CSV_DIR) and len(os.listdir(CSV_DIR)) > 0 else DEST_DIR
drive_csvs = [f for f in os.listdir(csv_folder) if f.lower().endswith('.csv')]

summary_data = []
for f in sorted(drive_csvs):
    f_path = os.path.join(csv_folder, f)
    size_mb = os.path.getsize(f_path) / (1024 * 1024)
    try:
        df_temp = pd.read_csv(f_path, nrows=5)
        with open(f_path, 'r', encoding='utf-8', errors='ignore') as fp:
            line_count = sum(1 for _ in fp) - 1
        cols_count = len(df_temp.columns)
        summary_data.append({
            "CSV File": f,
            "Size (MB)": f"{size_mb:.2f}",
            "Rows": f"{line_count:,}",
            "Columns": cols_count,
            "Drive Location": f_path
        })
    except Exception as e:
        summary_data.append({
            "CSV File": f,
            "Size (MB)": f"{size_mb:.2f}",
            "Rows": "N/A",
            "Columns": "N/A",
            "Drive Location": f_path
        })

df_summary = pd.DataFrame(summary_data)
try:
    from IPython.display import display
    display(df_summary)
except ImportError:
    print(df_summary.to_string())

print(f"\nTotal {len(drive_csvs)} CSV files securely saved and verified in Drive: {csv_folder}")